# Motivation Experiment — Symbolic vs. Semantic guidance under growing context

Run the cells **in order**. Every long-running cell is **resumable**: safe to re-run after a
Colab disconnect (it skips already-completed work). See `RUNBOOK.md` for what each go/no-go
gate looks like and what to do on each failure mode.

**Hard constraints:** T4 GPU only, fp16 everywhere (Turing SM 7.5: no bf16, no FlashAttention-2).


## Cell 1 — Setup (deps, repos, auth)
Pins deps **without touching Colab's preinstalled torch** (a torch swap breaks T4 CUDA).

In [ ]:

# --- Config you MUST set ---
SELF_REPO   = "https://github.com/PLACEHOLDER/dag-motivation-exp"  # <-- your repo
SELF_COMMIT = "main"                                              # <-- pin a commit for reproducibility
PRONTOQA_COMMIT = "PLACEHOLDER_PIN_ME"                            # <-- pin github.com/asaparov/prontoqa commit
DRIVE_DIR   = "/content/drive/MyDrive/motivation_exp"            # outputs live here (survive disconnects)

import subprocess, sys, os

# 1) Mount Drive (outputs + resume state persist here)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

# 2) Pin torch to the EXACT preinstalled version via a constraints file so nothing swaps it.
import torch
torch_ver = torch.__version__.split("+")[0]
with open("/content/constraints.txt", "w") as f:
    f.write(f"torch=={torch_ver}\n")
print("Colab torch:", torch.__version__, "| CUDA:", torch.version.cuda)

# 3) Clone + install the package (core is torch-free; [colab] extra adds xgrammar + s-t,
#    installed under the constraints file so pip cannot replace torch).
if not os.path.isdir("/content/dag-motivation-exp"):
    subprocess.run(["git", "clone", SELF_REPO, "/content/dag-motivation-exp"], check=True)
subprocess.run(["git", "-C", "/content/dag-motivation-exp", "checkout", SELF_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/content/constraints.txt",
                "-e", "/content/dag-motivation-exp[colab]"], check=True)

# 4) Fetch the pinned PrOntoQA generator.
subprocess.run(["bash", "/content/dag-motivation-exp/scripts/fetch_prontoqa.sh",
                "/content/prontoqa", PRONTOQA_COMMIT], check=True)

# 5) Hugging Face auth (Llama-3.2-3B is gated).
from huggingface_hub import login
login()  # paste a token with access to meta-llama/Llama-3.2-3B-Instruct

# 6) Sanity: nothing swapped torch.
import torch as _t; assert _t.__version__.split("+")[0] == torch_ver, "torch was changed by an install!"
print("Setup OK. torch intact:", _t.__version__)

## Cell 2 — Phase 0: Environment & smoke test
GPU assert, fp16 load, 8k generation with `logits_to_keep=1`, **peak-VRAM check**, eot-token assert.

In [ ]:

import torch, time, json, hashlib, subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from motivation_exp import config as C

# 1) T4 assert (all latency numbers must come from T4).
name = torch.cuda.get_device_name(0)
assert "T4" in name, f"Not a T4 (got {name!r}). Disconnect + retry (Runtime > Disconnect and delete runtime)."
print("GPU:", name)

# 2) Load model fp16 + SDPA (no bf16 / no FA2 on Turing).
tok = AutoTokenizer.from_pretrained(C.PRIMARY_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    C.PRIMARY_MODEL, torch_dtype=torch.float16, attn_implementation="sdpa", device_map="cuda",
)
model.eval()
print("vocab_size:", model.config.vocab_size)

# 3) Stop-token assert (Llama-3.2-Instruct stops on <|eot_id|>, NOT <|end_of_text|>).
eot_id = tok.convert_tokens_to_ids(C.STOP_TOKEN)
assert tok.convert_ids_to_tokens(eot_id) == C.STOP_TOKEN, "eot id mismatch"
print("eot id:", eot_id, "=", C.STOP_TOKEN)

# 4) Embedder fp16 on GPU.
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(C.EMBED_MODEL, device="cuda",
                               model_kwargs={"torch_dtype": torch.float16})
print("embedder loaded:", C.EMBED_MODEL)

# 5) 8k-context generation via the real backend (CHUNKED prefill) + peak-VRAM check (rev #1).
#    Chunked prefill is REQUIRED on T4: this GQA model always runs SDPA's math backend on sm75,
#    so a one-shot 8k prefill materializes a (1,24,8192,8192) fp32 score tensor and OOMs.
#    We exercise the exact code path the runs use (HFModelBackend), not a hand-rolled loop.
from motivation_exp import runner as R
torch.cuda.reset_peak_memory_stats()
filler = ("Every tumpus is a wumpus. " * 2000)
ids = tok(filler, return_tensors="pt").input_ids[:, :8192]
sync, sample_fn = R.make_torch_helpers()
backend = R.HFModelBackend(model, sync)
torch.cuda.synchronize(); t0 = time.perf_counter()
logits = backend.prefill(ids, chunk_size=C.PREFILL_CHUNK)   # incremental KV build
for _ in range(8):  # a few q_len=1 decode steps (score tensor is (1,24,1,L) -> trivial)
    nid = int(logits.argmax())
    logits = backend.decode_step(nid)
torch.cuda.synchronize()
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"8k smoke OK in {time.perf_counter()-t0:.1f}s | peak VRAM {peak_gb:.2f} GB "
      f"(chunk={C.PREFILL_CHUNK}; must be < ~15 GB)")
assert peak_gb < 15.5, "Still too high; lower config.PREFILL_CHUNK or cap the top bucket at 6k (RUNBOOK)."
del backend, logits; torch.cuda.empty_cache()

# 6) Env manifest.
freeze = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
env = {"gpu": name, "torch": torch.__version__, "cuda": torch.version.cuda,
       "vocab_size": model.config.vocab_size, "eot_id": eot_id, "peak_vram_gb_8k": peak_gb}
env["env_hash"] = hashlib.sha1(freeze.encode()).hexdigest()[:12]
with open(f"{DRIVE_DIR}/env_manifest.json", "w") as f: json.dump(env, f, indent=2)
with open(f"{DRIVE_DIR}/pip_freeze.txt", "w") as f: f.write(freeze)
print("env_hash:", env["env_hash"])

## Cell 3 — Phase 1: Data generation
Drives the PrOntoQA generator **through `_prontoqa_bridge`** (handles the clone-dir cwd for its
relative `bad_patterns.txt` open, and retries the generator's stochastic `(None,)*6` failures).
Each base item and each distractor-pool item gets its OWN disjoint concept block (synthetic
`*pus`-morphology names, registered with the generator's morphology), so every distractor is
concept-disjoint from every base item — the invariant the collision filter relies on. Reserves
the calibration + exemplar splits (disjoint from all buckets), buckets to token targets, runs the
validation gate, and writes `items.jsonl` + `manifest.json`.

In [ ]:

import json, os
from motivation_exp import datagen as dg, config as C
from motivation_exp._prontoqa_bridge import ProntoQABridge

br = ProntoQABridge("/content/prontoqa")

# One disjoint concept block per base item AND per pool item; register with the morphology.
reserved = br.reserved_nouns()
n_blocks = C.N_BASE_ITEMS + C.N_POOL_ITEMS
names = dg.make_concept_names(n_blocks * C.CONCEPT_BLOCK_SIZE, seed=C.GLOBAL_SEED, reserved=reserved)
br.register_concepts(names)
blocks = dg.partition_blocks(names, C.CONCEPT_BLOCK_SIZE)
base_blocks = blocks[:C.N_BASE_ITEMS]
pool_blocks = blocks[C.N_BASE_ITEMS:C.N_BASE_ITEMS + C.N_POOL_ITEMS]

def gen_items(block_list, prefix):
    items = []
    for i, block in enumerate(block_list):
        raw = br.generate(C.N_HOPS, block, ontology=C.ONTOLOGY, distractors=C.DISTRACTORS,
                          deduction_rule=C.DEDUCTION_RULE, no_adjectives=C.NO_ADJECTIVES)
        items.append(dg.raw_prontoqa_adapter(raw, item_id=f"{prefix}{i:04d}",
                                             base_id=f"{prefix}{i:04d}", concepts_hint=block))
    return items

bases = gen_items(base_blocks, "base")
pool_items = gen_items(pool_blocks, "pool")
pool = dg.build_distractor_pool(pool_items, size=20000, seed=C.GLOBAL_SEED)
print(f"generated {len(bases)} base + {len(pool_items)} pool items | pool sentences: {len(pool)}")

# Reserve disjoint calibration + exemplar splits BEFORE bucketing (R2 #1/#2).
bucket_bases, calib, exemplar = dg.reserve_splits(bases, C.N_CALIB_ITEMS, C.N_EXEMPLAR_ITEMS)

count_tokens = dg.hf_token_counter(tok)
buckets = dg.bucket_to_token_targets(bucket_bases, count_tokens, pool, C.BUCKETS, C.GLOBAL_SEED)

# Validation gate.
kept = {b: [] for b in C.BUCKETS}; dropped = 0
for b, its in buckets.items():
    for it in its:
        ok, reason = dg.validate_item(it, count_tokens)
        if ok: kept[b].append(it)
        else: dropped += 1; print("DROP", it.item_id, reason)
print("kept per bucket:", {b: len(v) for b, v in kept.items()}, "| dropped:", dropped)

# Persist to Drive.
data_dir = f"{DRIVE_DIR}/data"; os.makedirs(data_dir, exist_ok=True)
for b, its in kept.items():
    os.makedirs(f"{data_dir}/{b}", exist_ok=True)
    with open(f"{data_dir}/{b}/items.jsonl", "w") as f:
        for it in its: f.write(json.dumps(it.to_json()) + "\n")
for name, split in [("calib", calib), ("exemplar", exemplar)]:
    with open(f"{data_dir}/{name}.jsonl", "w") as f:
        for it in split: f.write(json.dumps(it.to_json()) + "\n")
dg.write_manifest(kept, f"{data_dir}/manifest.json", C.GLOBAL_SEED,
                  calib_ids=[c.base_id for c in calib], exemplar_ids=[e.base_id for e in exemplar],
                  extra={"vocab_isolation": "per-item disjoint concept blocks",
                         "concept_block_size": C.CONCEPT_BLOCK_SIZE, "n_pool_items": C.N_POOL_ITEMS,
                         "pool_sentences": len(pool), "no_adjectives": C.NO_ADJECTIVES,
                         "prontoqa_commit": C.PRONTOQA_COMMIT})
print("Phase 1 done ->", data_dir)

## Cell 4 — Phase 1.5: τ calibration + embedding discrimination test
Two parts on the 10 held-out calibration items:

1. **Discrimination test.** bge-small may not distinguish `X is slow` from `X is not slow`, nor
   one novel `*pus` concept from another (cosine can be > 0.9 for both). We measure raw
   cosine between each gold step and its polarity-flip / concept-swap. If these are high, the
   similarity search alone would false-accept them, so the checker's **predicate guard**
   (parsed-clause polarity+predicate exact match on the retrieved winner) is required — it is ON
   by default.
2. **τ sweep (guard ON — the reported config).** τ now controls retrieval *recall* (are gold
   steps retrieved from the growing candidate set); the guard supplies *precision* (negation /
   concept-swap are rejected). We report recall + false-accept rates with the guard ON, and ON vs
   OFF to show why the guard is needed, then freeze τ. No reported run uses uncalibrated τ (R2 #1).

In [ ]:

import numpy as np, random, json
from motivation_exp.checker import SemanticChecker, sentence_transformer_encoder
from motivation_exp import config as C, datagen as dg, grammar as gr

calib = dg.load_items(f"{DRIVE_DIR}/data/calib.jsonl")   # reload (resumable after disconnect)
encode_fn = sentence_transformer_encoder(embedder)
rng = random.Random(C.GLOBAL_SEED)
all_concepts = sorted({c for it in calib for c in it.concepts})

# --- 1) raw embedding discrimination (guard-independent) ---
def raw_cos(a, b):
    e = encode_fn([a, b]); return float(e[0] @ e[1])
neg_cos, swap_cos = [], []
for it in calib:
    for s in it.gold_steps:
        neg = gr.negate_sentence(s)
        if neg and neg != s: neg_cos.append(raw_cos(s, neg))
        others = [c for c in all_concepts if c not in s]
        if others:
            sw = gr.swap_concept(s, rng.choice(others))
            if sw and sw != s: swap_cos.append(raw_cos(s, sw))
if neg_cos:
    print(f"bge cos(step, negated):        median={np.median(neg_cos):.3f}  max={np.max(neg_cos):.3f}")
if swap_cos:
    print(f"bge cos(step, swapped-concept): median={np.median(swap_cos):.3f}  max={np.max(swap_cos):.3f}")
print("  -> if these are high (>~0.85) the embedding cannot separate them; the predicate guard "
      "(ON by default) is REQUIRED for restate precision.\n")

# --- 2) tau sweep: recall (gold accepted) + false-accept (neg/swap) with the guard on and off ---
def eval_tau(tau, guard):
    tp = n_pos = fp_neg = n_neg = fp_swap = n_swap = 0
    for it in calib:
        chk = SemanticChecker(encode_fn, tau, tau, predicate_guard=guard); chk.prefill(it.context)
        for s in it.gold_steps:
            tp += bool(chk.check_step(s).accepted); n_pos += 1        # recall (validated before accept)
            neg = gr.negate_sentence(s)
            if neg and neg != s: fp_neg += bool(chk.check_step(neg).accepted); n_neg += 1
            others = [c for c in all_concepts if c not in s]
            if others:
                sw = gr.swap_concept(s, rng.choice(others))
                if sw and sw != s: fp_swap += bool(chk.check_step(sw).accepted); n_swap += 1
            chk.accept(s)                                            # advance the true chain
    return (tp/max(1,n_pos), fp_neg/max(1,n_neg), fp_swap/max(1,n_swap))

print("tau sweep (guard ON — reported config):")
best = None
for tau in C.TAU_GRID:
    rec, fpn, fps = eval_tau(tau, guard=True)
    print(f"  tau={tau:.2f}  recall={rec:.2f}  FP_neg={fpn:.2f}  FP_swap={fps:.2f}")
    score = rec - 5 * (fpn + fps)   # prefer high recall, punish any false accept
    if best is None or score > best[1]: best = (tau, score)
print("\ntau sweep (guard OFF — demonstrates false accepts embeddings alone allow):")
for tau in C.TAU_GRID:
    rec, fpn, fps = eval_tau(tau, guard=False)
    print(f"  tau={tau:.2f}  recall={rec:.2f}  FP_neg={fpn:.2f}  FP_swap={fps:.2f}")

TAU = best[0]
print(f"\n>>> SELECTED tau = {TAU} (predicate_guard=True). "
      f"Paste into config.py:  TAU_RESTATE = {TAU}   TAU_MP = {TAU}")
mpath = f"{DRIVE_DIR}/data/manifest.json"; m = json.load(open(mpath))
m["frozen_tau"] = {"tau_restate": TAU, "tau_mp": TAU, "predicate_guard": True}
m["embedding_discrimination"] = {"neg_cos_median": float(np.median(neg_cos)) if neg_cos else None,
                                 "swap_cos_median": float(np.median(swap_cos)) if swap_cos else None}
json.dump(m, open(mpath, "w"), indent=2)

## Cell 5 — Phase 2: Pilot (20 items/bucket) + go/no-go gates
Runs all three methods interleaved. Resumable. Prints G1/G2/G3. **Freeze τ in `config.py`
(or set it below) before running** — the checker refuses uncalibrated τ.

In [ ]:

import json, itertools
from motivation_exp import runner as R, config as C, datagen as dg
from motivation_exp.grammar import make_compiler, synthesize_exemplar_cot
from motivation_exp.checker import SemanticChecker, sentence_transformer_encoder

# reload data from Drive (resumable after disconnect)
kept = {b: dg.load_items(f"{DRIVE_DIR}/data/{b}/items.jsonl") for b in C.BUCKETS}
exemplar = dg.load_items(f"{DRIVE_DIR}/data/exemplar.jsonl")

# --- set the frozen tau (paste from Phase 1.5 if not yet edited into config.py) ---
TAU_RESTATE = C.TAU_RESTATE if C.TAU_RESTATE is not None else TAU
TAU_MP      = C.TAU_MP      if C.TAU_MP      is not None else TAU

encode_fn = sentence_transformer_encoder(embedder)
compiler  = make_compiler(tok, model.config.vocab_size)
exemplar_item = exemplar[0]
exemplar_cot  = synthesize_exemplar_cot(exemplar_item)
cfg = R.DecodeCfg(eot_id=eot_id, max_new_tokens=C.MAX_NEW_TOKENS,
                  per_step_cap=C.PER_STEP_TOKEN_CAP, temp_schedule=tuple(C.TEMPERATURE_SCHEDULE),
                  max_retries=C.SEMANTIC_MAX_RETRIES, prefill_chunk_size=C.PREFILL_CHUNK)
sync, sample_fn = R.make_torch_helpers()
model_name = C.MODEL_SHORT[C.PRIMARY_MODEL]
env = json.load(open(f"{DRIVE_DIR}/env_manifest.json"))

def checker_factory():
    return SemanticChecker(encode_fn, TAU_RESTATE, TAU_MP, predicate_guard=C.PREDICATE_GUARD)

# take the first 20 items/bucket -> these are also the first 20 of the full run (rev #10)
pilot = {b: v[:C.PILOT_ITEMS_PER_BUCKET] for b, v in kept.items()}
pilot_out = f"{DRIVE_DIR}/results_pilot.jsonl"
R.run_all(model, tok, pilot, C.METHODS, pilot_out, model_name,
          exemplar_item=exemplar_item, exemplar_cot=exemplar_cot, cfg=cfg, sync=sync,
          sample_fn=sample_fn, compiler=compiler, checker_factory=checker_factory,
          gpu=env["gpu"], env_hash=env["env_hash"])

# ---- gates ----
from motivation_exp import plots
df = plots.load_results(pilot_out)
acc = plots._accuracy_table(df); lat = plots._latency_table(df)
g1 = acc.get("unguided", {}).get(min(C.BUCKETS), (0,))[0]
print(f"G1 unguided@{min(C.BUCKETS)} acc = {g1:.2f}  (want 0.40-0.80; <0.40 raise Qwen/hops=2, >0.80 hops=4)")
print("G2 (semantic >= symbolic, gap non-shrinking):")
for b in sorted(C.BUCKETS):
    s = acc.get("semantic", {}).get(b, (float('nan'),))[0]; y = acc.get("symbolic", {}).get(b, (float('nan'),))[0]
    print(f"   bucket {b}: semantic={s:.2f} symbolic={y:.2f} gap={s-y:+.2f}")
print("G3 (semantic latency rising, symbolic flat):")
for b in sorted(C.BUCKETS):
    s = lat.get("semantic", {}).get(b, (float('nan'),))[0]; y = lat.get("symbolic", {}).get(b, (float('nan'),))[0]
    print(f"   bucket {b}: semantic={s:.1f}ms symbolic={y:.1f}ms")

## Cell 6 — Phase 3: Full run (100 items/bucket, resumable)
Shares its output file with the pilot **by design (rev #10)**: the pilot's 20 completed
(model, method, bucket, item_id) tuples are skipped, not recomputed. Re-run this cell after
any disconnect until it prints `ALL DONE`.

In [ ]:

from motivation_exp import runner as R, config as C, datagen as dg

kept = {b: dg.load_items(f"{DRIVE_DIR}/data/{b}/items.jsonl") for b in C.BUCKETS}  # reload (resumable)
full = {b: v[:C.FULL_ITEMS_PER_BUCKET] for b, v in kept.items()}
full_out = f"{DRIVE_DIR}/results_full.jsonl"

# One-time: copy pilot rows into the full file so they count as completed (same items/seeds).
import os, shutil
if not os.path.exists(full_out) and os.path.exists(pilot_out):
    shutil.copyfile(pilot_out, full_out)

R.run_all(model, tok, full, C.METHODS, full_out, model_name,
          exemplar_item=exemplar_item, exemplar_cot=exemplar_cot, cfg=cfg, sync=sync,
          sample_fn=sample_fn, compiler=compiler, checker_factory=checker_factory,
          gpu=env["gpu"], env_hash=env["env_hash"])

# completion check
from motivation_exp import plots
done = plots.load_results(full_out)
need = len(C.BUCKETS) * C.FULL_ITEMS_PER_BUCKET * len(C.METHODS)
print(f"rows: {len(done)} / {need}")
print("ALL DONE" if len(done) >= need else "NOT DONE — re-run this cell after reconnecting.")

## Cell 7 — Phase 4: Two-panel motivation figure
Panel A accuracy (Wilson CIs), Panel B median per-token latency (IQR band). Exports PNG + PDF.

In [ ]:

from motivation_exp import plots, config as C
df = plots.load_results(f"{DRIVE_DIR}/results_full.jsonl")
out_png = f"{DRIVE_DIR}/motivation_figure.png"
plots.make_two_panel_figure(df, out_png, model_name=C.MODEL_SHORT[C.PRIMARY_MODEL])
print("wrote", out_png, "and .pdf")
from IPython.display import Image, display
display(Image(out_png))